<h3>Long-term trends</h3>
<p>The <code>site_trends()</code> function calculates historical AQ trends by site and pollutant (are pollution levels increasing or decreasing over multiple years)? It takes monthly averages of the input measurements and uses the seasonal Mann-Kendall and Theil-Sen tests to determine the direction, magnitude (slope), and significance of the trend, comparing each month to the same month in past years to account for seasonality.</p>
<p>The function applies data completeness criteria: the same month must have data for at least 3 different years, and for annual and network results, at least 9 months must have data during at least 3 years.</p>
<p><code>help(air.site_trends)</code> for more detail on the methodology.</p>

In [2]:
import airinsights as air
import plotly.graph_objects as go
import plotly.express as px

# load in long-term reference dataset for Detroit (from OpenAQ)
df, config = air.read_aqdata_file("sample_data/detroit_reference_openaq_2016to2026.csv.gz",
                                   config = "../config/detroit_openaq_config.yaml")

# run on no2, pm2.5 and so2
df = df[df['parameter'].isin(['no2','so2','pm25'])]

# convert units to ppb for interpretation
is_ppm = df["units"].str.lower() == "ppm"
df["value"] = df["value"].where(~is_ppm, df["value"] * 1000)
df["units"] = df["units"].where(~is_ppm, "ppb")

df.head()

Converting timestamp to local timezone: America/Detroit
Removed 30 exact duplicate rows


,datetime,parameter,value,units,sensors_id,location_id,lat,lon,site_name,site_locality,provider,monitor_type
0,2016-03-06 14:00:00-05:00,pm25,10.5,µg/m³,3855,2144,42.3231,-83.0689,DETROIT - W LAFAYETT,NaN,AirNow,reference
1,2016-03-06 15:00:00-05:00,pm25,11.6,µg/m³,3855,2144,42.3231,-83.0689,DETROIT - W LAFAYETT,NaN,AirNow,reference
2,2016-03-07 09:00:00-05:00,pm25,16.7,µg/m³,3855,2144,42.3231,-83.0689,DETROIT - W LAFAYETT,NaN,AirNow,reference
3,2016-03-07 10:00:00-05:00,pm25,12.1,µg/m³,3855,2144,42.3231,-83.0689,DETROIT - W LAFAYETT,NaN,AirNow,reference
4,2016-03-10 02:00:00-05:00,pm25,12.7,µg/m³,3855,2144,42.3231,-83.0689,DETROIT - W LAFAYETT,NaN,AirNow,reference


In [3]:
# run method, returning network, annual, and monthly trend results, plus the underlying monthly data (return_data=True)
network, annual, monthly, data = air.site_trends(df, config, return_data=True)

Skipping trends for no2 at site Eliza Howell Downwin: insufficient seasonal completeness
Skipping trends for no2 at site Windsor Downtown 1: insufficient seasonal completeness
Skipping trends for no2 at site Windsor West 1: insufficient seasonal completeness
Skipping trends for pm25 at site DETROIT - W LAFAYETT: insufficient seasonal completeness
Skipping trends for pm25 at site Detroit-E7 Mile: insufficient seasonal completeness
Skipping trends for pm25 at site Windsor West 1: insufficient seasonal completeness
Skipping trends for so2 at site Windsor Downtown 1: insufficient seasonal completeness
Skipping trends for so2 at site Windsor West 1: insufficient seasonal completeness
Skipping trends for so2 at site Windsor West 2: insufficient seasonal completeness


<h3>Network-level trends</h3>
<p><code>data['network_monthly']</code> has the network mean by month and pollutant, with the fitted trend line.</p>

In [4]:
# subset of network monthly mean data for NO2
network_data = data['network_monthly']
network_data = network_data[network_data['parameter'] == "no2"]
network_data.head()

,value,n_sites,parameter,trend_line
datetime,,,,
2016-03-01 00:00:00-05:00,NaN,3,no2,13.089216
2016-04-01 00:00:00-04:00,10.423291,3,no2,13.068337
2016-05-01 00:00:00-04:00,12.495345,3,no2,13.047457
2016-06-01 00:00:00-04:00,10.643418,3,no2,13.026578
2016-07-01 00:00:00-04:00,9.511437,3,no2,13.005699


Despite some data gaps and variability between years and seasons, the method detects a clear decreasing trend

In [5]:
# plot the network monthly mean and trend
fig = go.Figure([
    go.Scatter(x = network_data.index, y = network_data["value"], mode = "lines+markers", name = "Monthly mean"),
    go.Scatter(x = network_data.index, y = network_data["trend_line"], mode = "lines", line = dict(dash = "dash"), name = "Trend")
])
fig.update_layout(xaxis_title = "Date", yaxis_title = "NO2 (ppb)")
fig.show()

The results table shows that the network mean NO2 exhibited a significant (p < 0.05) decrease, with an estimated change of -0.25 ppb per year.

In [6]:
# show network mean trend results for no2
network[network['parameter'] == "no2"]

,parameter,trend,slope,intercept,p_value,n_sites_min,n_sites_median,n_sites_max,n_years,n_months,start_month,end_month
0,no2,decreasing,-0.250552,13.089216,0.000015,3,6.0,10,11,91,2016-04-01 00:00:00-04:00,2026-07-01 00:00:00-04:00


<h3>Annual trend by site</h3>
<p><code>annual</code> has one row per site and pollutant, with trend direction, slope, and significance (<code>p_value</code>) for the whole duration of the data.</p>

In [7]:
# preview of the annual trend results by site
annual.head()

,parameter,site_name,trend,slope,intercept,p_value,n_years,n_months,start_month,end_month,lat,lon
0,no2,Detroit-DP4th,decreasing,-0.218743,13.825380,3.080664e-02,9,64,2018-12-01 00:00:00-05:00,2026-07-01 00:00:00-04:00,42.312158,-83.091943
1,no2,Detroit-E7 Mile,decreasing,-0.242289,9.338250,9.758298e-07,11,69,2016-04-01 00:00:00-04:00,2026-07-01 00:00:00-04:00,42.430801,-83.000801
2,no2,Detroit-Military Par,no trend,-0.079905,11.907291,1.615926e-01,9,65,2018-12-01 00:00:00-05:00,2026-07-01 00:00:00-04:00,42.312078,-83.103469
3,no2,Detroit-Southwest,decreasing,-0.398360,13.992853,5.348119e-06,9,63,2018-07-01 00:00:00-04:00,2026-07-01 00:00:00-04:00,42.304200,-83.107200
4,no2,Detroit-Trinity,no trend,-0.143308,13.377092,1.164703e-01,9,65,2018-12-01 00:00:00-05:00,2026-07-01 00:00:00-04:00,42.295824,-83.129431


In [8]:
# heatmap of annual slope by pollutant and site. Asterisks mark sites and pollutants with significant trends.
trend = annual.pivot(index = "parameter", columns = "site_name", values = "slope")
pval = annual.pivot(index = "parameter", columns = "site_name", values = "p_value")
label = trend.round(1).astype(str) + pval.lt(0.05).map(lambda sig: " *" if sig else "")
label = label.mask(trend.isna(), "")
fig = px.imshow(trend, color_continuous_scale = "RdBu_r", color_continuous_midpoint = 0,
                 labels = dict(color = "slope"), title = "Annual trend by site (* = significant, p < 0.05)")
fig.update_traces(text = label, texttemplate = "%{text}")
fig.show()

<h3>Trends disaggregated by month</h3>


Now we can examine the trends during each month of the year for the site that showed the largest NO2 improvement.

While the trends at most months are not significant on their own, we can see that the greatest improvements for PM2.5 were in the winter, while one month (July) showed an increasing trend.

In [9]:
# heatmap of annual slope at "Detroit-Southwest" site by pollutant and month. Asterisks mark months and pollutants with significant trends.
site = "Detroit-Southwest"
site_monthly = monthly[monthly['site_name'] == site]
trend = site_monthly.pivot(index = "parameter", columns = "month", values = "slope")
pval = site_monthly.pivot(index = "parameter", columns = "month", values = "p_value")
label = trend.round(1).astype(str) + pval.lt(0.05).map(lambda sig: " *" if sig else "")
label = label.mask(trend.isna(), "")
fig = px.imshow(trend, color_continuous_scale = "RdBu_r", color_continuous_midpoint = 0,
                 labels = dict(color = "slope"), title = "Annual trend by site (* = significant, p < 0.05)")
fig.update_traces(text = label, texttemplate = "%{text}")
fig.show()

We can also plot the monthly mean data and trend line for the site to inspect it, this time for PM2.5.

In [10]:
# plot the site monthly mean and trend
site_data = data['site_monthly']
site_data = site_data[(site_data['site_name'] == site) & (site_data['parameter'] == "pm25")]
fig = go.Figure([
    go.Scatter(x = site_data.index, y = site_data["value"], mode = "lines+markers", name = "Monthly mean"),
    go.Scatter(x = site_data.index, y = site_data["trend_line"], mode = "lines", line = dict(dash = "dash"), name = "Trend")
])
fig.update_layout(xaxis_title = "Date", yaxis_title = "PM2.5 (ug/m3)")
fig.show()